# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第16章 機械学習・深層学習・Foundation Models ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch16_speaker_leakage.R`

対象章: 第16章「機械学習・深層学習・Foundation Models」

「教師あり学習の基礎と落とし穴」節の作例。話者holdoutを行わない
交差検証が、真の汎化性能をどれだけ過大評価するかを、架空のデータで
実際に確認する。実測データではない。

設計: 各話者に、ラベル(たとえば無声化の有無)についての固有の生起率
(話者ごとに安定した個人差)を持たせ、音響特徴量は主に話者固有の
「声質」を反映する(ラベルとの直接の対応は弱い)ものとして生成する。
こうすると、話者を特定できる特徴量を使うモデルは、訓練データに
含まれる話者については「この話者は何%の確率でラベル1を持つか」を
記憶するだけで高い精度を出せてしまうが、その記憶は新しい話者には
使えない。話者holdoutの有無で、この記憶による見かけの精度がどれだけ
過大評価かを確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(randomForest)

set.seed(20260906)

n_speakers    <- 20
n_tok_per_spk <- 40

speakers <- tibble(
  speaker      = sprintf("s%02d", 1:n_speakers),
  # 話者固有の「声質」オフセット。分散をトークン内ノイズ(SD 1)よりずっと
  # 大きく取ることで、この2つの特徴量だけからほぼ確実に話者を再識別できる
  # ようにする(リークが起きるための必要条件)。
  speaker_re   = rnorm(n_speakers, 0, 30),
  speaker_prob = runif(n_speakers, 0.1, 0.9)     # 話者ごとに安定したラベル生起率の個人差(大きめ)
)

d <- speakers |>
  tidyr::crossing(tibble(tok = 1:n_tok_per_spk)) |>
  mutate(
    label = factor(rbinom(n(), 1, speaker_prob)),               # 話者固有の生起率に従う
    label_num = as.numeric(as.character(label)),
    feature1 = 0.15 * label_num + speaker_re + rnorm(n(), 0, 1), # ラベルへの真の寄与は弱い
    feature2 = 0.1 * label_num + speaker_re * 0.8 + rnorm(n(), 0, 1)
  ) |>
  select(speaker, label, feature1, feature2)

# --- (a) 話者を考慮しない通常の交差検証(5-fold, ランダム分割) -----------
n_folds <- 5
d$fold_random <- sample(rep(1:n_folds, length.out = nrow(d)))

acc_random <- map_dbl(1:n_folds, function(k) {
  train <- d |> filter(fold_random != k)
  test  <- d |> filter(fold_random == k)
  m <- randomForest(label ~ feature1 + feature2, data = train, ntree = 200)
  mean(predict(m, test) == test$label)
})

# --- (b) 話者holdoutによる交差検証(話者ごとにfoldを割り当て) -----------
speaker_folds <- tibble(speaker = speakers$speaker,
                          fold_speaker = sample(rep(1:n_folds, length.out = n_speakers)))
d <- d |> left_join(speaker_folds, by = "speaker")

acc_speaker_holdout <- map_dbl(1:n_folds, function(k) {
  train <- d |> filter(fold_speaker != k)
  test  <- d |> filter(fold_speaker == k)
  m <- randomForest(label ~ feature1 + feature2, data = train, ntree = 200)
  mean(predict(m, test) == test$label)
})

cat("通常の交差検証(話者リークあり)の平均精度:", round(mean(acc_random) * 100, 1), "%\n")
cat("話者holdoutによる交差検証の平均精度:", round(mean(acc_speaker_holdout) * 100, 1), "%\n")
cat("最頻クラスによるベースライン:", round(max(prop.table(table(d$label))) * 100, 1), "%\n")

results <- tibble(
  method = c("通常の交差検証(話者リークあり)", "話者holdoutによる交差検証", "最頻クラスベースライン"),
  accuracy = c(mean(acc_random), mean(acc_speaker_holdout), max(prop.table(table(d$label))))
)

p <- ggplot(results, aes(x = reorder(method, accuracy), y = accuracy)) +
  geom_col(fill = "#34518A", width = 0.6) +
  geom_hline(yintercept = max(prop.table(table(d$label))), linetype = 2, colour = "#A8681F") +
  geom_text(aes(label = paste0(round(accuracy * 100, 1), "%")), hjust = -0.15, size = 3.5) +
  coord_flip(ylim = c(0, 1)) +
  scale_y_continuous(labels = scales::percent) +
  labs(x = NULL, y = "予測精度",
       caption = "シミュレーションデータ。破線は最頻クラスベースライン。scripts/R/ch16_speaker_leakage.R より生成。")
save_fig(p, "fig16-02-speaker-holdout", chapter = "ch16", width = 6.5, height = 3)
cat("完了。図はassets/figures/ch16/に保存した。\n")
